In [2]:
import zipfile
import unicodedata
import re
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# --- Step 1: Load and extract data from zip ---
def load_data_from_zip(zip_path, filename='spa.txt', num_sentences=None):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(filename) as file:
            lines = file.read().decode('utf-8').strip().split('\n')
            sentence_pairs = [line.split('\t')[:2] for line in lines if '\t' in line]
            if num_sentences:
                sentence_pairs = sentence_pairs[:num_sentences]
            return sentence_pairs

# --- Step 2: Normalize/clean sentences ---
def normalize(sentence):
    sentence = ''.join(c for c in unicodedata.normalize('NFD', sentence)
                       if unicodedata.category(c) != 'Mn')  # remove accents
    sentence = re.sub(r"[^a-zA-Z.!?¿]+", r" ", sentence)  # keep letters and punctuation
    sentence = re.sub(r"\s+", " ", sentence)  # remove extra spaces
    return sentence.lower().strip()

# --- Step 3: Preprocess sentence pairs ---
def preprocess_pairs(pairs):
    input_texts = [normalize(pair[0]) for pair in pairs]
    target_texts = ['<start> ' + normalize(pair[1]) + ' <end>' for pair in pairs]
    return input_texts, target_texts

# --- Step 4: Tokenization and Padding ---
def tokenize_and_pad(input_texts, target_texts):
    inp_tokenizer = Tokenizer(filters='')
    inp_tokenizer.fit_on_texts(input_texts)
    input_sequences = inp_tokenizer.texts_to_sequences(input_texts)
    max_len_inp = max(len(seq) for seq in input_sequences)
    input_sequences = pad_sequences(input_sequences, maxlen=max_len_inp, padding='post')

    tgt_tokenizer = Tokenizer(filters='')
    tgt_tokenizer.fit_on_texts(target_texts)
    target_sequences = tgt_tokenizer.texts_to_sequences(target_texts)
    max_len_tgt = max(len(seq) for seq in target_sequences)
    target_sequences = pad_sequences(target_sequences, maxlen=max_len_tgt, padding='post')

    return (input_sequences, target_sequences, max_len_inp, max_len_tgt,
            inp_tokenizer, tgt_tokenizer)

# --- Run preprocessing pipeline ---
# Assumes 'spa-eng.zip' is already uploaded to Colab environment
pairs = load_data_from_zip('spa-eng.zip', num_sentences=10000)
input_texts, target_texts = preprocess_pairs(pairs)
input_sequences, target_sequences, max_len_inp, max_len_tgt, inp_tokenizer, tgt_tokenizer = tokenize_and_pad(input_texts, target_texts)

# --- Split into model-ready inputs ---
encoder_input_data = np.array(input_sequences)
decoder_input_data = np.array([seq[:-1] for seq in target_sequences])
decoder_target_data = np.array([seq[1:] for seq in target_sequences])

# --- Vocabulary sizes ---
input_vocab_size = len(inp_tokenizer.word_index) + 1
target_vocab_size = len(tgt_tokenizer.word_index) + 1

# --- Quick sanity check ---
print("Sample Spanish:", input_texts[0])
print("Sample English:", target_texts[0])
print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)


Sample Spanish: go.
Sample English: <start> ve. <end>
Encoder input shape: (10000, 5)
Decoder input shape: (10000, 10)
Decoder target shape: (10000, 10)


In [6]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

# --- Define model parameters ---
embedding_dim = 256
units = 512  # LSTM hidden size

# --- Encoder ---
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(input_vocab_size, embedding_dim)(encoder_inputs)
encoder_lstm, state_h, state_c = LSTM(units, return_state=True)(enc_emb)
encoder_states = [state_h, state_c]

# --- Decoder ---
decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(target_vocab_size, embedding_dim)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(target_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# --- Define the full model ---
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# --- Compile the model ---
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# --- Train the model ---
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=5,
    validation_split=0.2
)


Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 299s 2s/step - loss: 3.6816 - val_loss: 2.3203
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 279s 2s/step - loss: 1.8070 - val_loss: 2.1601
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 331s 2s/step - loss: 1.6368 - val_loss: 2.0845
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 335s 2s/step - loss: 1.4843 - val_loss: 2.0087
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 325s 2s/step - loss: 1.3371 - val_loss: 1.9791


In [7]:
# Save the trained model
model.save("spanish_to_english_seq2seq.h5")

# Save tokenizers for later use
import pickle
with open('input_tokenizer.pkl', 'wb') as f:
    pickle.dump(inp_tokenizer, f)
with open('target_tokenizer.pkl', 'wb') as f:
    pickle.dump(tgt_tokenizer, f)


In [8]:
# Rebuild encoder model for inference
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference model
decoder_state_input_h = Input(shape=(units,))
decoder_state_input_c = Input(shape=(units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)

# Reverse lookup dictionaries
reverse_input_word_index = {i: word for word, i in inp_tokenizer.word_index.items()}
reverse_target_word_index = {i: word for word, i in tgt_tokenizer.word_index.items()}
target_word_index = tgt_tokenizer.word_index


In [9]:
def translate_sentence(input_sentence):
    # Preprocess input
    input_seq = inp_tokenizer.texts_to_sequences([normalize(input_sentence)])
    input_seq = pad_sequences(input_seq, maxlen=max_len_inp, padding='post')

    # Encode the input
    states_value = encoder_model.predict(input_seq)

    # Prepare <start> token as first decoder input
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_word_index['<start>']

    stop_condition = False
    decoded_sentence = ''

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index, '')

        if sampled_word == '<end>' or len(decoded_sentence.split()) > max_len_tgt:
            stop_condition = True
        else:
            decoded_sentence += ' ' + sampled_word

        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        states_value = [h, c]

    return decoded_sentence.strip()


In [10]:
!pip install gradio --quiet

import gradio as gr

# Define Gradio interface
interface = gr.Interface(
    fn=translate_sentence,
    inputs=gr.Textbox(lines=2, placeholder="Escribe una frase en español..."),
    outputs="text",
    title="Spanish to English Translator 🇪🇸→🇺🇸",
    description="Enter a sentence in Spanish and get its English translation."
)

interface.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ec2ea04ee537482e41.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
